[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/multiview/multiview_reconstruction.ipynb)

# Multi-View Reconstruction and Camera Alignment

Multi-view reconstruction and camera alignment recover the positions of observed landmarks and the poses of the cameras that see them from image pixels alone. Several cameras photograph the same landmarks from different places; each picture records only the direction toward a landmark, not how far away it lies. Where the sight lines from different cameras cross fixes the landmark, and when a camera's pose is misplaced, its sight lines miss the others' and the misses determine how to move the camera back into alignment.

A pixel's uncertainty on the sensor is a quadric ball, and feeding the camera into that ball and pulling its polar planes back through `camera.adjugate()` lifts it into a sight cone on scene points, `camera.adjugate()(ball(camera))`, which exists even though the camera loses a dimension. Sight cones add across views into a fused splat whose polar vanishes at the landmark, recovering its centre in closed form as `(splats + w * (w & Point)).solve(w)`, while Gauss-Newton alignment evaluates the cones' curvature and slope directly on the motion of the points under a camera twist.

In the matrix notation of projective vision, with a camera matrix $P$ acting on homogeneous points, planes pull back through $P^{\mathsf T}$, and a rank-one sensor precision $\mathbf n\, \mathbf n^{\mathsf T}$ lifts to the sight-cone matrix $P^{\mathsf T} \mathbf n\, \mathbf n^{\mathsf T} P$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D
from examples.util.animation import save_figure, show_animation
from examples.estimation.multiview import render

np.set_printoptions(precision=4, suppress=True)

# The multiview mathematics is written for any PGA; this notebook works in the plane.
ga = PGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Points are antivectors and planes vectors; a motor moves rigidly, and a twist
# is a small rigid step.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Motor = ga.gatype.rotor()                           # Rotor+
Twist = ga.gatype.bivector()                        # Bivector+
Quadric = ga.gatype((Plane, Point))
Information = ga.gatype((Scalar, Twist, Twist))
# The plane at infinity.
w = mv.w                                            # [] Plane+

## 1. The landmarks and the cameras

Six landmarks stand in front of three cameras: one on the left and one on the right, both turned inwards, and one between them looking straight ahead. Every camera is the same pinhole camera, carried to its place by its pose.

A pinhole camera sends a scene point to where the line from the point through the pinhole meets the sensor: the join of the pinhole with the point, met with the sensor plane, `(pinhole & point) ^ sensor`. Its action on every point, `(pinhole & Point) ^ sensor`, is the camera itself, a map from scene points to image points.

In [ ]:
# The landmarks, in front of the cameras.
landmark_coordinates = [[0.15, 0.85], [-0.43, 1.15], [0.50, 1.50], [0.03, 1.85], [0.65, 2.20], [-0.60, 2.55]]
# Each camera's offset along x and how far it is turned inwards: left, right, and in the middle.
camera_offsets = np.array([-0.75, 0.75, 0.0])
camera_turns = np.radians([18.0, -18.0, 0.0])

In [ ]:
landmarks = mv.point(landmark_coordinates)                                     # [landmarks] Point-
# Each camera moved along x, then turned.
rig = (mv.xw * (camera_offsets / 2)).exp() * (mv.xy * (camera_turns / 2)).exp()  # [cameras] Motor+

# The pinhole at the origin and the sensor, the plane one unit in front of it.
pinhole = mv.point([0, 0])                                                      # [] Point-
sensor = mv.y - mv.w                                                            # [] Plane+
camera = (pinhole & Point) ^ sensor                                             # [] Point- <- Point-
save_figure(render.draw_rig(rig), "multiview_rig");

## 2. Pixels and sight cones

Each camera sees each landmark at a pixel on its sensor. Because the pixel measurement is uncertain, it is represented by a quadratic cost around the pixel: the squared distance from the plane through the pixel across the sensor, written as its polarity map sending a point to its polar plane. A quadric's value at a point is its polar plane joined with the point.

Feeding the camera into that quadric and carrying its polar planes back through the camera lifts it into a sight cone on scene points, wider the further from the camera. What carries planes back is the camera's adjugate, the map on planes that incidence carries over from the camera, satisfying `camera.adjugate()(plane) & point == plane & camera(point)` for every plane and point even though the camera loses a dimension. A cone moves into the world like any map, `rig >> cones(rig << Point)`.

In the matrix notation of projective vision the adjugate reads as the transpose $P^{\mathsf T}$ of the camera matrix, through which planes back-project.

In [ ]:
# Every landmark as each camera sees it, in the camera's own frame, and its image on the sensor.
local_landmarks = rig << landmarks[:, None]                                     # [landmarks, cameras] Point-
images = camera(local_landmarks)                                                # [landmarks, cameras] Point-
pixels = images / (w & images)                                                  # [landmarks, cameras] Point-

# The plane through each pixel across the sensor, and the squared distance from it as a quadric.
pixel_planes = mv.x - mv.w * (mv.x & pixels)                                    # [landmarks, cameras] Plane+
balls = pixel_planes * (pixel_planes & Point)                                   # [landmarks, cameras] Plane+ <- Point-

# The camera fed into each ball, and its polar planes carried back: a sight cone in each camera's frame,
# and in the world.
back_projection = camera.adjugate()                                             # [] Plane+ <- Plane+
cones = back_projection(balls(camera))                                          # [landmarks, cameras] Plane+ <- Point-
world_cones = rig >> cones(rig << Point)                                        # [landmarks, cameras] Plane+ <- Point-
save_figure(render.draw_cones(rig, world_cones), "multiview_cones");

In [ ]:
print(back_projection)

## 3. Landmarks in closed form

The cones of one landmark from all cameras add into one quadric, a splat: compact where the sight lines cross steeply, and elongated where they are nearly parallel. Its centre is the landmark. At its centre the splat's polar vanishes, and adding a dyad on the plane at infinity, `w * (w & Point)`, makes the centre the pole of that plane, so the landmark is `(splats + w * (w & Point)).solve(w)`.

When the right camera is knocked out of place, the cones it contributes no longer cross the others' at the landmarks, and the fused splats spread.

In [ ]:
# The right camera knocked out of place: moved 70 cm along x and 30 cm along y, and turned 28 degrees.
knock_offsets = np.array([0.7, -0.3])
knock_turn = np.radians(-28.0)

In [ ]:
def triangulate(poses: Motor, cones: Quadric) -> tuple[Point, Quadric]:
    """The landmarks as the centres of their fused splats, found by carrying each landmark's sight cones into the world by the camera poses and summing across the cameras."""
    splats = (poses >> cones(poses << Point)).sum(axis=-1)                      # [landmarks] Plane+ <- Point-
    # The splat's polar vanishes at its centre; the dyad on the plane at infinity makes the centre its pole.
    centres = (splats + w * (w & Point)).solve(w).normalized()                  # [landmarks] Point-
    return centres, splats


def distance(points: Point) -> Scalar:
    """The root-mean-square distance of unit points from the true landmarks, from the dual norm of their ideal difference."""
    return (points - landmarks).dual().norm_squared().mean(axis=0).square_root()   # [] Scalar+


knock_motor = (((mv.xw * knock_offsets[0] - mv.yw * knock_offsets[1]) / 2).exp()
               * (mv.xy * (knock_turn / 2)).exp())                              # [] Motor+
# The rig as it is believed to be, with the right camera knocked.
guess = rig * stack([mv.rotor(), knock_motor, mv.rotor()])                      # [cameras] Motor+
initial_landmarks, initial_splats = triangulate(guess, cones)                   # [landmarks] Point-, [landmarks] Plane+ <- Point-
save_figure(render.draw_reconstruction(guess, guess >> cones(guess << Point), initial_splats, initial_landmarks),
            "multiview_guess");

In [ ]:
print(f"landmarks off by {(distance(initial_landmarks) * 100).to_array():.2f} cm with the right camera knocked")

## 4. Aligning the cameras

The cost of the scene is every landmark's cone values, summed over the cameras: the cone's polar of the landmark joined with the landmark. A small rigid step of a camera moves the landmarks in its frame by minus their commutator with the step, a linear map from steps to point motions. The polar of the motion joined with the motion is the cost's curvature over pairs of steps, and the landmark's polar joined with the motion is its slope; because the cone is already the cost, no residual vector is formed. Solving the curvature against the slope gives each camera's step, and alternating that step with placing the landmarks at their splats' centres walks the knocked camera back into place. The other two cameras stay where they are: pictures alone cannot tell the whole scene and rig moved together, and the anchored cameras fix where the scene lies. Each step closes about a sixth of what remains because the landmarks move with the camera between steps, which a step of the camera alone does not account for; stretching the step and capping its reach moves the camera at an even pace from far off until it settles. The multiview core module provides two further alignment methods: one joint step of cameras and landmarks together, `align_jointly`, and steps that bring each pixel's sight line through the other cameras' splat of its landmark without placing a landmark, `align_rays_to_splats`.

In the matrix notation of least squares the curvature and slope read as $J^{\mathsf T} J$ and $J^{\mathsf T} \mathbf r$, with $J$ the Jacobian of the residual $r = \mathbf n^{\mathsf T} \mathbf p$ of the single-plane balls; here no Jacobian is formed, and for a quadric cost with no residual behind it the same two expressions evaluate its curvature and slope.

In [ ]:
# The cameras that move, the knocked one; how far a step is stretched, the length it is capped at, and the depth
# of the landmarks, in metres, at which a turn counts as much as a move; the steps shown, and the duration of a
# frame in milliseconds.
free_cameras = np.array([0.0, 1.0, 0.0])
stretch, step_cap, depth = 4.0, 0.08, 1.5
n_steps = 30
frame_duration = 120

In [ ]:
def alignment(poses: Motor, cones: Quadric, free_cameras: np.ndarray) -> Iterator[tuple[Motor, Point, Quadric, Information]]:
    """The camera poses, triangulated landmarks, fused splats, and cost curvature at each alternating Gauss-Newton step."""
    while True:
        centres, splats = triangulate(poses, cones)
        # The landmarks in each camera's frame, and how a small step of the camera moves them.
        local_centres = poses << centres[:, None]                               # [landmarks, cameras] Point-
        motion = -Twist.commutator(local_centres)                               # [landmarks, cameras] Point- <- Twist+
        # The cone's polar of the motion joined with the motion is the curvature over pairs of steps; the
        # landmark's polar joined with the motion is the slope.
        curvature = (cones(motion) & motion).sum(axis=0)                        # [cameras] Scalar+ <- (Twist+, Twist+)
        slope = (cones(local_centres) & motion).sum(axis=0)                     # [cameras] Scalar+ <- Twist+
        yield poses, centres, splats, curvature
        # Each moving camera's step, stretched and capped; the others stay where they are.
        step = curvature.solve(-slope) * free_cameras * stretch                 # [cameras] Twist+
        step_size = (((step / 2).exp() >> pinhole) - pinhole).dual().norm() + depth * step.norm()   # [cameras] Scalar+
        poses = poses * (step * (step_cap / (step_size + step_cap)) / 2).exp()        # [cameras] Motor+


def cost(poses: Motor, centres: Point) -> Scalar:
    """The total sight-cone cost of the landmarks summed across all cameras."""
    local_centres = poses << centres[:, None]                                   # [landmarks, cameras] Point-
    return (cones(local_centres) & local_centres).sum()                         # [] Scalar+


estimates = list(islice(alignment(guess, cones, free_cameras), n_steps + 1))
show_animation(render.animate_convergence((poses, poses >> cones(poses << Point), splats, centres)
                                          for poses, centres, splats, _ in estimates),
               "multiview_reconstruction_convergence", frame_duration, stepped=True)

In [ ]:
for index, (poses, centres, _, _) in islice(enumerate(estimates), None, None, 5):
    print(f"step {index}: landmarks off by {(distance(centres) * 100).to_array():.3f} cm, cost {cost(poses, centres).to_array():.3e}")

## 5. The cameras' pose uncertainty

The curvature of the cost over a camera's steps, divided by the variance of a pixel, is the information on its pose: how sharply the log-likelihood falls as the camera moves each way. Its inverse is the pose covariance, drawn at three standard deviations as the three-sigma region of the camera's position and three standard deviations of its heading either way (one standard deviation, a few centimetres and about a degree and a half, would be too small to see at the scale of the scene). Each camera's information is its own curvature with the other cameras held in place. Carried through the motion of the pinhole under a step, the covariance gives the uncertainty of the camera's position, a map from planes to points whose inverse about the pinhole is the ellipsoid drawn, while the heading's variance is the covariance read along the turn rate.

In statistical terms the information reads as the Fisher information $\mathcal{I}$, and the covariance as $\mathcal{I}^{-1}$.

In [ ]:
# The pixel's noise, one standard deviation on the sensor one unit in front of the pinhole: as wide as the
# cones are drawn.
pixel_noise = 0.035
# The standard deviations the uncertainties are drawn at, and the camera's own forward direction, along its axis.
sigmas = 3
optical_axis = mv.direction([0, 1])                                             # [] Direction-

In [ ]:
poses, centres, splats, curvature = estimates[-1]
# The information on each camera's pose, with the others held in place: its curvature over the pixel's variance.
information = curvature / pixel_noise**2                                        # [cameras] Scalar+ <- (Twist+, Twist+)
# How a step of each camera moves its pinhole in the world, and how it turns the camera's heading, read across the
# camera's axis.
pinholes = poses >> pinhole                                                     # [cameras] Point-
pinhole_motion = poses >> Twist.commutator(pinhole)                             # [cameras] Point- <- Twist+
heading_rate = mv.x.regressive(Twist.commutator(optical_axis))                  # [] Scalar+ <- Twist+
# The covariance carried through the pinhole's motion, at sigmas standard deviations, about the pinhole: the
# uncertainty of each camera's position, a map from planes to points, and its inverse, the ellipsoid.
uncertainty = (pinhole_motion(information.solve(Plane & pinhole_motion)) * sigmas**2
               - pinholes * (Plane & pinholes))                                 # [cameras] Point- <- Plane+
ellipsoids = uncertainty.inverse()                                              # [cameras] Plane+ <- Point-
# The heading's spread: the covariance read along the turn.
spreads = heading_rate(information.solve(heading_rate)).square_root() * sigmas  # [cameras] Scalar+
save_figure(render.draw_reconstruction_with_covariance(poses, poses >> cones(poses << Point), splats, centres,
                                                        ellipsoids, spreads), "multiview_covariance");

In [ ]:
# checks
from examples import instantiate

core = instantiate("examples.estimation.multiview.core", ga)
# The landmarks seen from the true rig are found exactly; the knocked camera spreads them; the alignment
# brings them back, and the cost falls to round-off.
true_centres, true_splats = triangulate(rig, cones)
core_centres, core_splats = core.triangulate(rig, cones)
np.testing.assert_allclose((true_centres - core_centres).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((true_splats - core_splats).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(distance(true_centres).to_array(), 0.0, atol=1e-9)
assert distance(initial_landmarks) > 1e-3
np.testing.assert_allclose(distance(centres).to_array(), 0.0, atol=1e-3)
assert cost(*estimates[-1][:2]) < 1e-3 * cost(*estimates[0][:2])